# 14 · FlashAttention and the Online Softmax

Companion to **Chapter 17**. You will prove to yourself that the online softmax is
*exact*, implement tiled attention, and measure the memory scaling that makes
long context possible.

In [ ]:
import math
import torch
import torch.nn.functional as F
torch.manual_seed(0)

## 1 · Why you cannot just stream `exp(x).sum()`

Softmax needs a global max for numerical stability. Watch what happens without it.

In [ ]:
x = torch.tensor([100.0, 200.0, 300.0, 400.0])
print("naive exp(x).sum():      ", torch.exp(x).sum().item())
print("stable (subtract max):   ", (x.max() + torch.exp(x - x.max()).sum().log()).item())
print("torch.logsumexp:         ", torch.logsumexp(x, 0).item())
print("\nexp(400) overflows fp32 (max ~3.4e38, exp overflows above x~88).")
print("Every real softmax subtracts the max. Which needs to SEE the whole row.")

## 2 · The online softmax — exact, in one pass

Keep a running max `m` and running sum `l`; rescale when the max changes.

In [ ]:
def online_logsumexp(xs, chunk=8, verbose=False):
    m, l = float('-inf'), 0.0
    for i in range(0, len(xs), chunk):
        blk = xs[i:i+chunk]
        m_new = max(m, blk.max().item())
        corr = math.exp(m - m_new) if m != float('-inf') else 0.0
        l = l * corr + torch.exp(blk - m_new).sum().item()
        if verbose:
            print(f"  block {i//chunk}: m {m:8.3f} -> {m_new:8.3f}   "
                  f"rescale {corr:.6f}   l = {l:.6f}")
        m = m_new
    return m + math.log(l)

xs = torch.tensor([1.0, 3.0, 2.0, 5.0, 4.0, 0.5, 6.0, 2.5,
                   9.0, 1.5, 3.5, 7.0, 2.0, 8.0, 0.0, 4.5])
print("streaming, block by block:")
streaming = online_logsumexp(xs, chunk=4, verbose=True)
oneshot = torch.logsumexp(xs, 0).item()
print(f"\nstreaming logsumexp: {streaming:.10f}")
print(f"one-shot  logsumexp: {oneshot:.10f}")
print(f"difference:          {abs(streaming-oneshot):.2e}")
assert abs(streaming - oneshot) < 1e-6
print("\nEXACT, not approximate. That equality is the theorem behind FlashAttention.")

In [ ]:
# Stress it: adversarial magnitudes, many chunk sizes
print(f"{'scale':>8} {'chunk':>7} {'abs error':>12}")
for scale in [1.0, 50.0, 500.0]:
    for chunk in [1, 7, 64]:
        v = torch.randn(257) * scale
        err = abs(online_logsumexp(v, chunk) - torch.logsumexp(v, 0).item())
        print(f"{scale:>8.0f} {chunk:>7} {err:>12.2e}")
        assert err < 1e-3
print("\nExact at every scale and every block size.")

## 3 · Exercise 16.2 — tiled attention

The full algorithm. Slower than naive in Python (no SRAM control), but
**numerically identical** — which is the point.

In [ ]:
def naive_attention(Q, K, V, causal=True):
    N, d = Q.shape
    S = Q @ K.T / math.sqrt(d)                       # the (N, N) matrix
    if causal:
        S = S.masked_fill(torch.ones(N, N, dtype=torch.bool).triu(1), float('-inf'))
    return S.softmax(-1) @ V


def flash_attention(Q, K, V, Br=32, Bc=32, causal=True):
    """The (N,N) score matrix is NEVER materialised -- only (Br, Bc) tiles."""
    N, d = Q.shape
    O = torch.zeros(N, d)

    for i in range(0, N, Br):                        # outer: query blocks
        Qi = Q[i:i+Br]
        Oi = torch.zeros(Qi.size(0), d)              # running output
        mi = torch.full((Qi.size(0),), float('-inf'))# running max
        li = torch.zeros(Qi.size(0))                 # running sum

        for j in range(0, N, Bc):                    # inner: key blocks
            if causal and j > i + Qi.size(0) - 1:
                break                                # whole block is masked; skip
            Kj, Vj = K[j:j+Bc], V[j:j+Bc]
            Sij = Qi @ Kj.T / math.sqrt(d)           # (Br, Bc) <- the ONLY tile
            if causal:
                qi = torch.arange(i, i+Qi.size(0))[:, None]
                kj = torch.arange(j, j+Kj.size(0))[None, :]
                Sij = Sij.masked_fill(kj > qi, float('-inf'))

            m_new = torch.maximum(mi, Sij.max(dim=-1).values)
            corr  = torch.exp(mi - m_new)            # the rescale factor
            Pij   = torch.exp(Sij - m_new[:, None])
            li = corr * li + Pij.sum(-1)             # rescale + accumulate denom
            Oi = corr[:, None] * Oi + Pij @ Vj       # rescale + accumulate numer
            mi = m_new

        O[i:i+Br] = Oi / li[:, None]                 # normalise ONCE at the end
    return O


N, d = 128, 32
Q, K, V = torch.randn(N, d), torch.randn(N, d), torch.randn(N, d)
ref = naive_attention(Q, K, V)

print(f"{'Br':>5} {'Bc':>5} {'max abs error':>15}")
for Br, Bc in [(16, 16), (32, 32), (64, 16), (N, N)]:
    got = flash_attention(Q, K, V, Br, Bc)
    err = (got - ref).abs().max().item()
    print(f"{Br:>5} {Bc:>5} {err:>15.2e}")
    assert torch.allclose(got, ref, atol=1e-5)
print("\nIdentical at every tile size ✓  Tiling changes nothing about the answer.")

## 4 · The memory scaling (Exercise 16.3)

The slope of this curve is the whole point.

In [ ]:
def score_matrix_bytes(T, n_head=32, dtype=2):
    return n_head * T * T * dtype

def flash_bytes(T, n_head=32, d=128, dtype=2):
    return n_head * T * d * dtype * 3          # Q, K, V only -- O(N)

def fmt(b):
    for u in ["B","KB","MB","GB","TB"]:
        if b < 1024: return f"{b:7.2f} {u}"
        b /= 1024
    return f"{b:.1f} PB"

print(f"{'T':>8} {'naive scores':>14} {'flash Q/K/V':>14} {'ratio':>9}")
prev_T = prev_naive = None
for T in [512, 1024, 4096, 16384, 65536, 131072]:
    nb, fb = score_matrix_bytes(T), flash_bytes(T)
    print(f"{T:>8} {fmt(nb):>14} {fmt(fb):>14} {nb/fb:>8.0f}x")
    if prev_T:
        assert abs((nb/prev_naive) / (T/prev_T)**2 - 1) < 1e-6   # exactly quadratic
    prev_T, prev_naive = T, nb

print("\nnaive grows as T^2 (slope 2 on log-log); flash grows as T (slope 1).")
print("At T=131072 the naive score tensor for ONE layer is larger than any GPU.")
print("\nDIAGNOSTIC: if your 'flash' memory scales quadratically, you fell back to")
print("the math backend. Most common cause: your tensors are fp32, not bf16/fp16.")

## 5 · What PyTorch actually dispatches to

In [ ]:
q = torch.randn(1, 8, 256, 64)
k = torch.randn(1, 8, 256, 64)
v = torch.randn(1, 8, 256, 64)

out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print(f"F.scaled_dot_product_attention -> {tuple(out.shape)}")

# verify it equals our tiled version for one head
mine = flash_attention(q[0,0], k[0,0], v[0,0], 32, 32)
print(f"matches our tiled implementation: "
      f"{torch.allclose(mine, out[0,0], atol=1e-4)} ✓")

print(f"\nRunning on: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print("On CPU there is no FlashAttention kernel -- PyTorch uses a math/`flash`-style")
print("CPU path. The ALGORITHM is what matters here; the kernel is a GPU concern.")
print("\nOn a CUDA box, force/inspect a backend with:")
print("  from torch.nn.attention import sdpa_kernel, SDPBackend")
print("  with sdpa_kernel(SDPBackend.FLASH_ATTENTION): ...")

---
## Self-check

1. FlashAttention does the same (or slightly more) arithmetic. Why is it faster?
2. What does the correction factor `exp(m_old - m_new)` do, and why is it exact?
3. Why does the backward pass recompute attention weights rather than store them?
4. Does FlashAttention reduce the KV cache?

<details><summary>Answers</summary>

1. Attention is bottlenecked by HBM traffic, not FLOPs. Not materialising the
   `(N,N)` matrix removes a huge round-trip to slow memory.
2. It re-bases everything accumulated so far from `e^{-m_old}` to `e^{-m_new}`.
   Numerator and denominator get the *same* factor, so their ratio — the softmax
   output — is unchanged. Algebra, not approximation.
3. Storing an `(N,N)` matrix would reintroduce exactly the `O(N²)` memory and
   traffic the design exists to eliminate.
4. **No.** That is GQA/MLA's job (Chapter 16). Flash speeds up *computing*
   attention; it doesn't shrink what you *store*. They are complementary and are
   always used together.

</details>

**Next:** `23_dpo.ipynb`